# Train ResNet50 and EfficientNet-B0 on real-scan slices (Colab)

1. **Runtime > Change runtime type > T4 GPU.**
2. Put `data_real.zip` (from the Kaggle notebook `nfti-to-jpeg-conversion.ipynb`) in Google Drive under `MyDrive/brain_tumor/`.
3. Run all cells.

The two models train one after the other in the same run. Checkpoints, curves, confusion matrices and metrics are written to `MyDrive/brain_tumor/outputs/`, so they survive a Colab disconnect. If the runtime drops, just re-run all cells: a model that already finished training is skipped.

After training, the **shortcut checks** at the bottom test whether the models are reading the tumor or just recognising which dataset a slice came from.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/brain_tumor"
DATA_ZIP = f"{DRIVE_DIR}/data_real.zip"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"
MODELS = ["resnet50", "efficientnet_b0"]
HEAD_EPOCHS, EPOCHS, BATCH_SIZE = 5, 10, 32

In [ ]:
# Get the code. If the repo is private, upload the repo folder to Drive instead
# and set REPO_DIR to it.
BRANCH = "main"
REPO_DIR = "/content/repo"
%cd /content
!rm -rf {REPO_DIR}
!git clone --depth 1 -b {BRANCH} https://github.com/imfonggg/AI-for-Brain-Tumor-Detection.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# Copy the slices to local disk (much faster to read than Drive during training)
import os
%cd {REPO_DIR}
!rm -rf /content/data_real
!unzip -q {DATA_ZIP} -d /content/data_real
!find /content/data_real -maxdepth 2 -type d

# Send outputs/ straight to Drive
os.makedirs(OUTPUT_DIR, exist_ok=True)
!rm -rf {REPO_DIR}/outputs
!ln -s {OUTPUT_DIR} {REPO_DIR}/outputs

In [ ]:
# Train each model, then evaluate it on the held-out Testing/ patients.
# The curves image is written at the very end of train.py, so it marks a finished run.
# A finished run is reused only if it was trained on the same classes as the current data.
import torch
%cd {REPO_DIR}
data_classes = sorted(os.listdir("/content/data_real/Training"))


def finished_on_same_classes(model):
    if not (os.path.exists(f"outputs/{model}_curves.png") and os.path.exists(f"outputs/{model}_best.pt")):
        return False
    ckpt_classes = torch.load(f"outputs/{model}_best.pt", map_location="cpu").get("class_names")
    if ckpt_classes != data_classes:
        print(f"{model}: saved run used classes {ckpt_classes}, data has {data_classes}; retraining")
        return False
    return True


for model in MODELS:
    if finished_on_same_classes(model):
        print(f"{model}: already trained on {data_classes}, skipping (delete outputs/{model}_curves.png to retrain)")
    else:
        !python train.py --model {model} --data_dir /content/data_real \
            --head_epochs {HEAD_EPOCHS} --epochs {EPOCHS} --batch_size {BATCH_SIZE}
    !python evaluate.py --checkpoint outputs/{model}_best.pt --data_dir /content/data_real \
        --output_csv outputs/{model}_metrics.csv

In [ ]:
# Side-by-side comparison
%cd {REPO_DIR}
csvs = " ".join(f"outputs/{m}_metrics.csv" for m in MODELS)
!python compare_models.py {csvs}

from IPython.display import Image, display
display(Image("outputs/model_comparison.png"))
for model in MODELS:
    display(Image(f"outputs/{model}_curves.png"))
    display(Image(f"outputs/{model}_confusion_matrix.png"))

## Shortcut checks: is the model reading the tumor?

Every class comes from a different dataset (BraTS glioma, BraTS meningioma, IXI healthy scans, Figshare metastases), so the models could score well by recognising the *dataset* (brightness, skull or no skull, borders) instead of the tumor. A perfect score on one class is a warning sign, not proof of success. These cells test for that, using the trained models in `outputs/`. Run the setup cell first, then any of the three checks. Each check saves its figure or table to `outputs/` on Drive.

1. **Grad-CAM:** heatmaps of where each model looks, 8 test slices per class. Good: the heat sits on the tumor (or spread over brain tissue for no tumor). Bad: the heat sits on the skull, the image edge or the black background.
2. **Source test:** a simple classifier that only sees plain image statistics (brightness histogram, contrast, how much black border) tries to guess which dataset each slice came from. If it gets close to 100%, the datasets are easy to tell apart without looking at any tumor, so a shortcut is *available* to the CNNs.
3. **Masked test:** re-runs each model on altered test slices. If a class is still recognised after the brain center is blanked out, after the pixels are shuffled, or after the model's own hot spot is erased, the model is not relying on the tumor for that class.

In [ ]:
# ---------- Shortcut checks: setup (run once before the three checks) ----------
import glob, os, random, re
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

REPO_DIR = "/content/repo"
DATA_DIR = "/content/data_real"
OUT = "outputs"
MODELS = ["resnet50", "efficientnet_b0"]
%cd {REPO_DIR}
from data_loader import get_transforms
from model import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_tf = get_transforms(train=False)
CLASSES = sorted(os.listdir(f"{DATA_DIR}/Testing"))


def load_model(name):
    ckpt = torch.load(f"{OUT}/{name}_best.pt", map_location=device)
    model = build_model(name, num_classes=len(ckpt["class_names"]), pretrained=False)
    model.load_state_dict(ckpt["model_state_dict"])
    return model.to(device).eval(), ckpt["class_names"]


def target_layer(model, name):
    """Last convolutional block, where Grad-CAM is usually taken."""
    return model.layer4[-1] if name.startswith("resnet") else model.features[-1]


def sample_test_files(per_class, seed=0):
    rng = random.Random(seed)
    files = {}
    for cls in CLASSES:
        paths = sorted(glob.glob(f"{DATA_DIR}/Testing/{cls}/*.jpg"))
        files[cls] = rng.sample(paths, min(per_class, len(paths)))
    return files


def load_gray(path):
    """Grayscale uint8 array at the model's input size."""
    return np.asarray(Image.open(path).convert("L").resize((224, 224)), dtype=np.uint8)


def to_batch(gray_images):
    return torch.stack([eval_tf(Image.fromarray(g).convert("RGB")) for g in gray_images]).to(device)


@torch.no_grad()
def predict(model, gray_images, batch_size=32):
    preds = []
    for i in range(0, len(gray_images), batch_size):
        preds.append(model(to_batch(gray_images[i:i + batch_size])).argmax(1).cpu())
    return torch.cat(preds).numpy()


def gradcam(model, name, gray):
    """Grad-CAM heatmap (224x224, 0..1) for the predicted class, plus that class index and confidence."""
    store = {}

    def hook(_module, _inputs, output):
        store["act"] = output
        output.register_hook(lambda g: store.setdefault("grad", g))

    handle = target_layer(model, name).register_forward_hook(hook)
    model.zero_grad(set_to_none=True)
    logits = model(to_batch([gray]))
    idx = int(logits.argmax(1))
    logits[0, idx].backward()
    handle.remove()
    weights = store["grad"].mean(dim=(2, 3), keepdim=True)
    cam = (weights * store["act"]).sum(1, keepdim=True).relu()
    cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[0, 0]
    cam = cam.detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, idx, torch.softmax(logits, 1)[0, idx].item()


missing = [m for m in MODELS if not os.path.exists(f"{OUT}/{m}_best.pt")]
if missing:
    raise FileNotFoundError(f"No trained checkpoint for {missing} in {OUT}/. Run the training cell first.")
print("classes:", CLASSES, "| test slices:", {c: len(os.listdir(f"{DATA_DIR}/Testing/{c}")) for c in CLASSES})

In [ ]:
# ---------- Check 1: Grad-CAM, 8 test slices per class, both models ----------
if "load_model" not in globals():
    raise RuntimeError("Run the shortcut-checks setup cell first.")

PER_CLASS = 8
files = sample_test_files(PER_CLASS, seed=1)
for name in MODELS:
    model, class_names = load_model(name)
    fig, axes = plt.subplots(len(CLASSES), PER_CLASS, figsize=(2.2 * PER_CLASS, 2.5 * len(CLASSES)),
                             squeeze=False)
    background_heat = {}
    for row, cls in enumerate(CLASSES):
        shares = []
        for col, path in enumerate(files[cls]):
            gray = load_gray(path)
            cam, idx, conf = gradcam(model, name, gray)
            # Share of the heat that falls on near-black pixels (background / outside the brain)
            shares.append(cam[gray < 10].sum() / (cam.sum() + 1e-8))
            ax = axes[row, col]
            ax.imshow(gray, cmap="gray")
            ax.imshow(cam, cmap="jet", alpha=0.4)
            ok = "" if class_names[idx] == cls else "  (WRONG)"
            ax.set_title(f"true {cls}\npred {class_names[idx]} {conf:.2f}{ok}", fontsize=7)
        for ax in axes[row]:
            ax.axis("off")
        background_heat[cls] = float(np.mean(shares)) if shares else float("nan")
    fig.suptitle(f"Grad-CAM: {name}", fontsize=12)
    fig.tight_layout()
    path = f"{OUT}/gradcam_{name}.png"
    fig.savefig(path, dpi=110)
    plt.show()
    print(f"{name}: saved {path}")
    print("  share of heat on black background (lower is better):",
          {c: f"{v:.0%}" for c, v in background_heat.items()})

In [ ]:
# ---------- Check 2: can plain image statistics tell the datasets apart? ----------
if "load_model" not in globals():
    raise RuntimeError("Run the shortcut-checks setup cell first.")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

CLASS_SOURCE = {"glioma": "BraTS glioma", "meningioma": "BraTS meningioma",
                "metastasis": "Figshare metastasis", "notumor": "IXI healthy", "pituitary": "pituitary set"}
MAX_PER_SOURCE = 1500   # per split, keeps this check to a minute or two


def source_of(path):
    cls = os.path.basename(os.path.dirname(path))
    if cls == "notumor" and not os.path.basename(path).startswith("IXI"):
        return "BraTS tumor-free"            # from --notumor_from_tumor_patients
    return CLASS_SOURCE.get(cls, cls)


def features(path):
    """Statistics a dataset leaves behind regardless of tumor: brightness, contrast, black border, edges."""
    g = np.asarray(Image.open(path).convert("L").resize((128, 128)), dtype=np.float32)
    rows, cols = np.where(g > 10)
    box = ([rows.min(), rows.max(), cols.min(), cols.max()] if rows.size else [0, 0, 0, 0])
    ring = np.concatenate([g[:8].ravel(), g[-8:].ravel(), g[:, :8].ravel(), g[:, -8:].ravel()])
    hist = np.histogram(g, bins=32, range=(0, 255))[0] / g.size
    return np.concatenate([[g.mean(), g.std(), (g < 10).mean(), (g > 245).mean(), ring.mean(),
                            np.abs(np.diff(g, axis=1)).mean()],
                           np.percentile(g, [1, 5, 25, 50, 75, 95, 99]), np.array(box) / 128, hist])


def load_split(split, seed):
    paths = glob.glob(f"{DATA_DIR}/{split}/*/*.jpg")
    by_source = {}
    for p in paths:
        by_source.setdefault(source_of(p), []).append(p)
    rng = random.Random(seed)
    chosen = [p for ps in by_source.values() for p in rng.sample(ps, min(MAX_PER_SOURCE, len(ps)))]
    return np.stack([features(p) for p in chosen]), np.array([source_of(p) for p in chosen])


X_train, y_train = load_split("Training", seed=0)
X_test, y_test = load_split("Testing", seed=0)
clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0).fit(X_train, y_train)
pred = clf.predict(X_test)
labels = sorted(set(y_test.tolist()))
acc = (pred == y_test).mean()
print(f"Source accuracy from image statistics alone: {acc:.1%}  (chance is about {1 / len(labels):.0%})\n")
print(classification_report(y_test, pred, zero_division=0))
print("rows = true source, columns = guessed source:", labels)
print(confusion_matrix(y_test, pred, labels=labels))
with open(f"{OUT}/source_test.txt", "w") as f:
    f.write(f"accuracy {acc:.4f}\n" + classification_report(y_test, pred, zero_division=0))
if acc > 0.95:
    print("\nVerdict: the datasets are almost perfectly separable without looking at any tumor. "
          "A shortcut is available; check Grad-CAM and the masked test to see whether the CNNs use it.")
elif acc > 0.75:
    print("\nVerdict: the datasets are fairly easy to tell apart from image statistics. Some shortcut risk.")
else:
    print("\nVerdict: image statistics alone do not identify the dataset well. Low shortcut risk from these cues.")

In [ ]:
# ---------- Check 3: does each model still recognise a class with the tumor area removed? ----------
if "load_model" not in globals():
    raise RuntimeError("Run the shortcut-checks setup cell first.")
import pandas as pd

PER_CLASS = 50
files = sample_test_files(PER_CLASS, seed=2)
yy, xx = np.mgrid[:224, :224]
CENTER = ((yy - 112) / 80.0) ** 2 + ((xx - 112) / 80.0) ** 2 <= 1   # central ellipse, ~40% of the image


def fill_value(gray):
    brain = gray[gray > 10]
    return int(np.median(brain)) if brain.size else 0


def blank_center(gray):
    """Keep only the outer ring (edges, skull, background)."""
    out = gray.copy()
    out[CENTER] = 0
    return out


def shuffle_pixels(gray, rng):
    """Same brightness histogram, no shapes at all."""
    return rng.permutation(gray.ravel()).reshape(gray.shape)


def blank_hotspot(gray, cam, top=0.10):
    """Erase the 10% of pixels the model looks at most, filling with typical brain intensity."""
    out = gray.copy()
    out[cam >= np.quantile(cam, 1 - top)] = fill_value(gray)
    return out


rows = []
for name in MODELS:
    model, class_names = load_model(name)
    rng = np.random.default_rng(0)
    for cls in CLASSES:
        if cls not in class_names:
            continue
        target = class_names.index(cls)
        grays = [load_gray(p) for p in files[cls]]
        variants = {
            "original": grays,
            "center blanked": [blank_center(g) for g in grays],
            "pixels shuffled": [shuffle_pixels(g, rng) for g in grays],
            "hot spot erased": [blank_hotspot(g, gradcam(model, name, g)[0]) for g in grays],
        }
        row = {"model": name, "class": cls, "n": len(grays)}
        for label, imgs in variants.items():
            row[label] = float((predict(model, imgs) == target).mean())
        rows.append(row)

table = pd.DataFrame(rows)
table.to_csv(f"{OUT}/masked_test.csv", index=False)
pd.options.display.float_format = "{:.0%}".format
display(table)
print("Each number is how often the model still predicts the TRUE class after the change.")
print("For a model that reads the tumor: 'original' high, and the other three columns clearly lower")
print("for the tumor classes. A tumor class that stays high with the center blanked or pixels")
print("shuffled is being recognised from edges, skull or brightness instead of the tumor.")
fig, ax = plt.subplots(figsize=(10, 3.5))
table.set_index(["model", "class"])[["original", "center blanked", "pixels shuffled", "hot spot erased"]] \
    .plot.bar(ax=ax, ylim=(0, 1), rot=30)
ax.set_ylabel("still predicts true class")
ax.set_title("Masked test")
fig.tight_layout()
fig.savefig(f"{OUT}/masked_test.png", dpi=110)
plt.show()